In [1]:
import os
import re
import json
import shutil
import pyarrow.parquet as pq
import pandas as pd

INPUT_FILE = "acl-publication-info.74k.v2.parquet"
OUT_DIR = "acl_fulltext_docs"
META_FILE = "acl_fulltext_meta.jsonl"

MAX_DOCS = 40000
MIN_WORDS = 3000
BATCH_SIZE = 200

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

if os.path.exists(META_FILE):
    os.remove(META_FILE)

def clean_text(text):
    if text is None or pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def word_count(text):
    return len(text.split()) if text else 0

parquet_file = pq.ParquetFile(INPUT_FILE)

saved = 0
skipped_empty = 0
skipped_short = 0

with open(META_FILE, "w", encoding="utf-8") as meta_out:
    for batch in parquet_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=["title", "abstract", "full_text", "acl_id", "year", "author", "url"]
    ):
        rows = batch.to_pylist()

        for row in rows:
            title = clean_text(row.get("title"))
            abstract = clean_text(row.get("abstract"))
            full_text = clean_text(row.get("full_text"))
            acl_id = clean_text(row.get("acl_id"))
            year = clean_text(row.get("year"))
            author = clean_text(row.get("author"))
            url = clean_text(row.get("url"))

            if not full_text:
                skipped_empty += 1
                continue

            wc = word_count(full_text)
            if wc < MIN_WORDS:
                skipped_short += 1
                continue

            doc_text = f"Title: {title}\n\nAbstract: {abstract}\n\nFull text:\n{full_text}\n"

            filename = f"doc_{saved:06d}.txt"
            filepath = os.path.join(OUT_DIR, filename)

            with open(filepath, "w", encoding="utf-8") as out:
                out.write(doc_text)

            meta = {
                "doc_id": saved,
                "file": filename,
                "acl_id": acl_id,
                "title": title,
                "author": author,
                "year": year,
                "url": url,
                "word_count": wc,
                "text_length": len(doc_text)
            }
            meta_out.write(json.dumps(meta, ensure_ascii=False) + "\n")

            saved += 1

            if saved % 500 == 0:
                print(f"Сохранено документов: {saved}")

            if saved >= MAX_DOCS:
                break

        if saved >= MAX_DOCS:
            break

print()
print("Готово.")
print(f"Сохранено документов: {saved}")
print(f"Пропущено пустых full_text: {skipped_empty}")
print(f"Пропущено коротких (< {MIN_WORDS} слов): {skipped_short}")
print(f"Папка корпуса: {OUT_DIR}")
print(f"Метафайл: {META_FILE}")

Сохранено документов: 500
Сохранено документов: 1000
Сохранено документов: 1500
Сохранено документов: 2000
Сохранено документов: 2500
Сохранено документов: 3000
Сохранено документов: 3500
Сохранено документов: 4000
Сохранено документов: 4500
Сохранено документов: 5000
Сохранено документов: 5500
Сохранено документов: 6000
Сохранено документов: 6500
Сохранено документов: 7000
Сохранено документов: 7500
Сохранено документов: 8000
Сохранено документов: 8500
Сохранено документов: 9000
Сохранено документов: 9500
Сохранено документов: 10000
Сохранено документов: 10500
Сохранено документов: 11000
Сохранено документов: 11500
Сохранено документов: 12000
Сохранено документов: 12500
Сохранено документов: 13000
Сохранено документов: 13500
Сохранено документов: 14000
Сохранено документов: 14500
Сохранено документов: 15000
Сохранено документов: 15500
Сохранено документов: 16000
Сохранено документов: 16500
Сохранено документов: 17000
Сохранено документов: 17500
Сохранено документов: 18000
Сохранено до

In [ ]:
pip install pandas pyarrow

In [3]:
import os
import json

RAW_FILE = "acl-publication-info.74k.v2.parquet"
DOCS_DIR = "acl_fulltext_docs"
META_FILE = "acl_fulltext_meta.jsonl"

raw_size = os.path.getsize(RAW_FILE)

doc_count = 0
text_bytes = 0
text_chars = 0
text_words = 0

for name in os.listdir(DOCS_DIR):
    path = os.path.join(DOCS_DIR, name)
    if os.path.isfile(path) and name.endswith(".txt"):
        doc_count += 1
        text_bytes += os.path.getsize(path)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
            text_chars += len(text)
            text_words += len(text.split())

avg_doc_size = text_bytes / doc_count if doc_count else 0
avg_doc_chars = text_chars / doc_count if doc_count else 0
avg_doc_words = text_words / doc_count if doc_count else 0

print("Размер сырых данных (байт):", raw_size)
print("Количество документов:", doc_count)
print("Размер выделенного текста (байт):", text_bytes)
print("Средний размер документа (байт):", avg_doc_size)
print("Средний объём текста в документе (символов):", avg_doc_chars)
print("Средний объём текста в документе (слов):", avg_doc_words)

Размер сырых данных (байт): 514785389
Количество документов: 40000
Размер выделенного текста (байт): 1201818038
Средний размер документа (байт): 30045.45095
Средний объём текста в документе (символов): 29826.697525
Средний объём текста в документе (слов): 4954.905075
